# Online Shoppers EC2 Large Experiment

- Experiment slug: `online-shoppers-ec2-large`
- Problem type: `classification`
- Primary workflow: `both`
- Metrics to optimize:
- `pr_auc`
- `f1`
- `brier_score`
- Notebook path: `notebook/`
- Validation: 5-fold CV
- Expected close-out: metrics table plus champion model


## Experiment metadata

In [ ]:
import json
from pathlib import Path

EXPERIMENT_SLUG = "online-shoppers-ec2-large"
ROOT = Path.cwd()
if not (ROOT / "reports").exists():
    ROOT = ROOT.parent
DATASET_PATH = ROOT / "data/raw/online_shoppers_intention.csv"
COMPARISON_PATH = ROOT / "reports/experiments/final_model_comparison.json"
METRICS_PATH = ROOT / "reports/model_metrics.json"
PROTOCOL_PATH = ROOT / "reports/experiments/protocol_manifest.json"
TARGET_COLUMN = "Revenue"
PRIMARY_METRIC = "pr_auc"
SECONDARY_METRICS = ["f1", "brier_score"]
PROBLEM_TYPE = "classification"
RUN_MODE = "both"
N_SPLITS = 5
RANDOM_STATE = 42

## Reproducible setup

- Keep `N_SPLITS = 5`.
- Define the winning direction of the primary metric before comparing models.
- If you change the validation strategy, justify it in markdown.


In [ ]:
import random

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
comparison = json.loads(COMPARISON_PATH.read_text(encoding="utf-8"))
metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
protocol

## Dataset loading and target definition

Load the dataset, inspect shape and dtypes, and confirm the target column.

In [ ]:
df = pd.read_csv(DATASET_PATH)
assert TARGET_COLUMN in df.columns
assert len(df) == protocol["dataset_rows"]
df.head()

## Baseline

Build the first simple baseline before feature engineering so later gains are measurable.

In [ ]:
baseline = next(
    candidate
    for candidate in comparison["candidates"]
    if candidate["name"] == "dummy__engineered_with_page_values__prior"
)
baseline["metrics"]

## Feature engineering log

The campaign compared the same 33 configurations with and without `PageValues`. The engineered transformer adds duration-per-page ratios, total activity and duration, exit/bounce gaps, administrative and informational shares, logarithmic count/duration features, and cyclical month signals. The aggregate below quantifies the effect.

In [ ]:
feature_set_summary = (
    pd.DataFrame(
        {
            "feature_set": candidate["feature_set"],
            "cv_pr_auc_mean": candidate["metrics"]["cv_pr_auc_mean"],
        }
        for candidate in comparison["candidates"]
    )
    .groupby("feature_set", as_index=False)
    .agg(best_cv_pr_auc=("cv_pr_auc_mean", "max"))
    .sort_values("best_cv_pr_auc", ascending=False)
)
feature_set_summary

## Experiment tracker

Every candidate was stored as an individual MLflow child run on EC2. This notebook reads the versioned comparison export so the complete 66-run leaderboard remains reproducible.

In [ ]:
experiment_rows = [
    {
        "name": candidate["name"],
        "family": candidate["family"],
        "feature_set": candidate["feature_set"],
        "cv_mean": candidate["metrics"]["cv_pr_auc_mean"],
        "cv_std": candidate["metrics"]["cv_pr_auc_std"],
        "f1": candidate["metrics"]["oof_f1"],
        "brier_score": candidate["metrics"]["oof_brier_score"],
        "params": candidate["params"],
        "run_id": candidate["run_id"],
    }
    for candidate in comparison["candidates"]
]


def leaderboard(ascending=True):
    board = pd.DataFrame(experiment_rows)
    if board.empty:
        return board
    return board.sort_values(
        ["cv_mean", "cv_std"],
        ascending=[ascending, True],
    ).reset_index(drop=True)

## CatBoost-first boosting experiments

Start with CatBoost. Add alternative boosters only when they create a meaningful comparison.

In [ ]:
catboost_runs = leaderboard(ascending=False).query("family == 'catboost'")
catboost_runs.head(10)

## Alternative boosters

Use this section for XGBoost, LightGBM, HistGradientBoosting, or other justified baselines.

In [ ]:
booster_runs = leaderboard(ascending=False).query(
    "family in ['xgboost', 'lightgbm', 'hist_gradient_boosting']"
)
booster_runs.head(10)

## PyTorch experiments

Three PyTorch MLP architectures were evaluated per feature set. The requested execution environment was Linux EC2, where Apple MPS is unavailable, so training used the CPU fallback.

In [ ]:
try:
    import torch
except ImportError:
    torch = None

if torch is None:
    device = "missing-torch"
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Selected device: {device}")

In [ ]:
pytorch_runs = leaderboard(ascending=False).query("family == 'pytorch_mlp'")
pytorch_runs

## Leaderboard and error analysis

Review the full experiment table, not just the best score. Explain failures and regressions.

In [ ]:
assert len(experiment_rows) == 66
assert not comparison["failures"]
leaderboard(ascending=False).head(15)

## Champion model summary

Close every run with the winning model, chosen metrics, CV detail, and the next best experiment.

In [ ]:
winner = comparison["candidates"][0]
runner_up = comparison["candidates"][1]
winner_name = winner["name"]
winner_family = winner["family"]
winner_score = winner["metrics"]["cv_pr_auc_mean"]
winner_secondary_metrics = {
    "cv_f1_mean": winner["metrics"]["cv_f1_mean"],
    "cv_brier_score_mean": winner["metrics"]["cv_brier_score_mean"],
    "audit_pr_auc": metrics["test"]["pr_auc"],
    "audit_f1": metrics["test"]["f1"],
}
winner_feature_blocks = winner["feature_set"]
winner_params = winner["params"]
winner_reason = (
    "Highest mean PR-AUC across 5 group-aware folds; no candidate failures. "
    f"Runner-up: {runner_up['name']} ({runner_up['metrics']['cv_pr_auc_mean']:.6f})."
)
next_iteration = "Monitor drift and recalibrate the threshold on new labeled sessions."

print(f"Winner model: {winner_name}")
print(f"Family: {winner_family}")
print(f"Primary metric: {PRIMARY_METRIC} = {winner_score}")
print(f"Secondary metrics: {winner_secondary_metrics}")
print(f"Key features: {winner_feature_blocks}")
print(f"Key params: {winner_params}")
print(f"Why it won: {winner_reason}")
print(f"Next iteration: {next_iteration}")